# Phase 1 LoRA Training (Kaggle T4×2)

**Run this notebook on Kaggle** with GPU (T4×2). Do not run locally unless you have the same data layout.- **Input**: Dataset linked as input (e.g. `souissiyoussef/diffusion-text2image`) → `/kaggle/input/datasets/souissiyoussef/diffusion-text2image/data`
- **Output**: `results/` (checkpoints, metrics, visualizations, reports)Ensure the dataset contains `data/processed/train/`, `data/processed/val/`, and that **src/** (config, dataset, train, evaluate, visualize, mlflow_utils) is available (e.g. in the same Kaggle dataset or copied into the notebook).

##  Charger la Configuration et les Modules

### Objectif:
Importer tous les modules du projet et vérifier les chemins.

### Ce que fait cette cellule:
- Charge config.py, dataset.py, train.py
- Vérifie que les chemins sont corrects

In [2]:
# Remove Kaggle's preinstalled xformers (broken: aoti_torch_create_device_guard / No module named 'xformers.ops')
!pip uninstall -y xformers 2>/dev/null || true

# Install mlflow (not preinstalled on Kaggle)
!pip install -q mlflow

# Inject fake xformers with valid __spec__ so diffusers can load (we use use_xformers=False in training)
import sys
import importlib.util
spec_x = importlib.util.spec_from_loader("xformers", loader=None, origin="fake")
spec_ops = importlib.util.spec_from_loader("xformers.ops", loader=None, origin="fake")
xformers_fake = importlib.util.module_from_spec(spec_x)
xformers_ops_fake = importlib.util.module_from_spec(spec_ops)
xformers_fake.ops = xformers_ops_fake
sys.modules["xformers"] = xformers_fake
sys.modules["xformers.ops"] = xformers_ops_fake
print("Fake xformers installed in sys.modules")

Fake xformers installed in sys.modules


## Charger et Inspecter le Dataset CelebA

### Objectif:
Charger les 182,339 images du dataset CelebA.

### Ce que fait cette cellule:
- Charge les métadonnées (train et val)
- Affiche les statistiques
- Valide les chemins des images

In [3]:
import sys
import importlib.util
from pathlib import Path

# Remove any existing xformers
for key in list(sys.modules.keys()):
    if key == "xformers" or key.startswith("xformers."):
        del sys.modules[key]

# Install fake xformers
spec_x = importlib.util.spec_from_loader("xformers", loader=None, origin="fake")
spec_ops = importlib.util.spec_from_loader("xformers.ops", loader=None, origin="fake")
xformers_fake = importlib.util.module_from_spec(spec_x)
xformers_ops_fake = importlib.util.module_from_spec(spec_ops)
xformers_fake.ops = xformers_ops_fake
sys.modules["xformers"] = xformers_fake
sys.modules["xformers.ops"] = xformers_ops_fake

# Kaggle paths
KAGGLE_DATASET = Path("/kaggle/input/datasets/souissiyoussef/diffusion-text2image")
project_root = str(KAGGLE_DATASET) if (KAGGLE_DATASET / "src" / "config.py").exists() else None

if project_root and project_root not in sys.path:
    sys.path.insert(0, project_root)
    sys.path.insert(0, str(KAGGLE_DATASET / "src"))
    
print("Project root for imports:", project_root or "current dir")

# ONLY import what we need for training
from config import TrainConfig, get_processed_root, get_results_root, get_celeba_img_dir
from dataset import Text2ImageDataset
from train import run_training
from visualize import plot_training_history, plot_fid_results, plot_dataset_distribution
from mlflow_utils import ExperimentTracker

Project root for imports: /kaggle/input/datasets/souissiyoussef/diffusion-text2image
🎯 Environment: KAGGLE
   Data (metadata) root: /kaggle/input/datasets/souissiyoussef/diffusion-text2image/data/processed
   CelebA root: /kaggle/input/datasets/jessicali9530/celeba-dataset/img_align_celeba
   Images dir: /kaggle/input/datasets/jessicali9530/celeba-dataset/img_align_celeba/img_align_celeba


2026-03-08 01:01:21.676383: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772931681.874798      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772931681.930335      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772931682.418532      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772931682.418571      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772931682.418574      23 computation_placer.cc:177] computation placer alr

## LANCER L'ENTRAÎNEMENT LoRA 

### Objectif:
Entraîner le modèle pendant 5,000 ou 15,000 étapes selon votre configuration.

### Ce que fait cette cellule:
- Lance la boucle d'entraînement complète
- Sauvegarde les checkpoints automatiquement
- Enregistre les métriques (loss, learning rate)
- Suivi via MLflow

### ⏱️ Durée estimée: 1-3 heures

### 📊 Métriques à surveiller:
- **Loss**: Doit diminuer progressivement
- **Learning Rate**: Augmente pendant warmup, puis constant
- **ETA**: Temps estimé restant

In [4]:
data_root = get_processed_root()
print("Data root:", data_root)
print("Exists:", data_root.exists() if data_root else False)
train_meta = Path(data_root) / "train" / "metadata.csv" if data_root else None
print("train/metadata.csv exists:", train_meta.exists() if train_meta else False)
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

Data root: /kaggle/input/datasets/souissiyoussef/diffusion-text2image/data/processed
Exists: True
train/metadata.csv exists: True
CUDA available: True
Device: Tesla T4


## Charger le Modèle Entraîné avec LoRA

### Objectif:
Charger le modèle avec les poids LoRA entraînés.

### Ce que fait cette cellule:
- Charge Stable Diffusion v1.5
- Charge les poids LoRA depuis le checkpoint final
- Vérifie que LoRA est correctement chargé

In [5]:
train_dataset = Text2ImageDataset("train", data_root=get_processed_root(), validate=False)
val_dataset = Text2ImageDataset("val", data_root=get_processed_root(), validate=False)
print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")
if len(train_dataset) > 0:
    s = train_dataset[0]
    print(f"Sample domain: {s['domain']}, class: {s['class_name']}")

Train samples: 182338
Val samples: 20261
Sample domain: celeba, class: visage_humain


## Générer des Images de Test

### Objectif:
Générer des images avec le modèle entraîné pour évaluer la qualité.

### Ce que fait cette cellule:
- Définit des prompts de test
- Génère une image pour chaque prompt
- Sauvegarde les images générées

In [6]:
# Kaggle T4 settings — aligned with Run 1 (checkpoint-15000, best loss=0.023538)
TrainConfig.batch_size = 1
TrainConfig.gradient_accumulation_steps = 4
TrainConfig.max_train_steps = 15000
TrainConfig.save_steps = 2500
TrainConfig.logging_steps = 100
TrainConfig.fp16 = True
TrainConfig.use_xformers = False
TrainConfig.gradient_checkpointing = True
TrainConfig.resolve_paths()
print("Output dir:", TrainConfig.output_dir)
print("Logging dir:", TrainConfig.logging_dir)

Output dir: /kaggle/working/results/checkpoints
Logging dir: /kaggle/working/results/logs


In [7]:
# Pre-training checklist - all should show OK
import torch
checks = []
# GPU
if torch.cuda.is_available():
    checks.append(("GPU available", True, torch.cuda.get_device_name(0)))
else:
    checks.append(("GPU available", False, "No GPU"))
# Data
data_root = get_processed_root()
meta = Path(data_root) / "train" / "metadata.csv"
checks.append(("Data loads", meta.exists(), str(data_root)))
# Model dirs writable
out_dir = Path(TrainConfig.output_dir)
out_dir.mkdir(parents=True, exist_ok=True)
test_file = out_dir / ".write_test"
try:
    test_file.write_text("ok")
    test_file.unlink()
    checks.append(("Checkpoints dir writable", True, TrainConfig.output_dir))
except Exception as e:
    checks.append(("Checkpoints dir writable", False, str(e)))
# Config
checks.append(("max_train_steps == 15000", TrainConfig.max_train_steps == 15000, f"current: {TrainConfig.max_train_steps}"))

for name, ok, detail in checks:
    status = "OK" if ok else "FAIL"
    print(f"  [{status}] {name}: {detail}")
if not all(c[1] for c in checks):
    raise RuntimeError("Some checks failed. Fix before running training.")
print("All checks passed. Ready to run continuation training (25,000 steps total, ~7.8h).")

  [OK] GPU available: Tesla T4
  [OK] Data loads: /kaggle/input/datasets/souissiyoussef/diffusion-text2image/data/processed
  [OK] Checkpoints dir writable: /kaggle/working/results/checkpoints
  [OK] max_train_steps == 25000: current: 25000
All checks passed. Ready to run continuation training (25,000 steps total, ~7.8h).


In [8]:
try:
    pipe, exp_tracker = run_training(TrainConfig)
    summary = exp_tracker.get_summary()
    print("Final loss:", summary.get("final_loss"))
    print("Total steps:", summary.get("total_steps"))
    print("Elapsed (s):", summary.get("elapsed_seconds"))
except Exception as e:
    import traceback
    print("Training failed:", e)
    traceback.print_exc()
    raise

Checkpoint trouve : /kaggle/input/datasets/souissiyoussef/diffusion-text2image/output/results/checkpoints/checkpoint-15000
Reprise depuis checkpoint : /kaggle/input/datasets/souissiyoussef/diffusion-text2image/output/results/checkpoints/checkpoint-15000
Entrainement sur : cuda


2026/03/08 01:01:44 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/08 01:01:44 INFO mlflow.store.db.utils: Updating database tables
2026/03/08 01:01:46 INFO mlflow.tracking.fluent: Experiment with name 'stable-diffusion-lora' does not exist. Creating a new experiment.


Chargement du dataset...
Echantillons d'entrainement : 182338
DataLoader : 182338 batches
Chargement du modele de base : stabilityai/stable-diffusion-xl-base-1.0


config.json:   0%|          | 0.00/631 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

...

unet/diffusion_pytorch_model.fp16.safete(…):   0%|          | 0.00/5.14G [00:00<?, ?B/s]

vae_1_0/diffusion_pytorch_model.fp16.saf(…):   0%|          | 0.00/167M [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Reprise LoRA depuis checkpoint : /kaggle/input/datasets/souissiyoussef/diffusion-text2image/output/results/checkpoints/checkpoint-15000
NOTE : nouvelle config LoRA (to_k + to_out.0 + dropout=0.1).
       Les poids to_q/to_v sont charges depuis le checkpoint.
       Les poids to_k/to_out.0 sont initialises a zero (gaussian).
OK Checkpoint LoRA charge (UNet brut, is_trainable=True)
Parametres UNet total    : 2,573,269,764
Parametres LoRA trainab. : 5,806,080
Parametres entrainables pour l'optimizer : 5,806,080
AdamW standard utilise (use_8bit_adam=False)
[Resume] global_step = 15000 — steps restants : 10000
Demarrage entrainement — max_steps=25000
Batch effectif : 4

Epoque 1
[Etape 15000/25000] Loss: 0.003007 | LR: 0.00e+00 | ETA: 9.7h
[Etape 15100/25000] Loss: 0.011315 | LR: 1.00e-05 | ETA: 12.1h
[Etape 15200/25000] Loss: 0.235835 | LR: 2.00e-05 | ETA: 12.0h
[Etape 15300/25000] Loss: 0.005193 | LR: 3.00e-05 | ETA: 11.9h
[Etape 15400/25000] Loss: 0.156473 | LR: 4.00e-05 | ETA: 11.8h
[Et

## FID EVALUATION

In [9]:
# FID: CelebA only (single domain evaluation)
import numpy as np
from pathlib import Path
import json

print("\n" + "="*70)
print("FID EVALUATION - CelebA Single Domain")
print("="*70)

# Load validation dataset
print("\nLoading validation dataset for FID reference...")
val_dataset = Text2ImageDataset("val", validate=False)
print(f"Validation images available: {len(val_dataset)}")

# Load generated images
qual_dir = Path("/kaggle/input/datasets/souissiyoussef/diffusion-text2image/output/results/qualitative/celeba")
gen_images = list(qual_dir.glob("*.png"))
print(f"Generated images: {len(gen_images)}")

if len(gen_images) >= 10 and len(val_dataset) >= 50:
    print("\n✓ Sufficient images for FID calculation")
    
    try:
        # Import here to avoid issues
        from evaluate import extract_features_with_inception, compute_fid_from_features
        
        # Get real image paths
        real_image_paths = []
        for sample in val_dataset.samples[:500]:
            image_filename = sample["image_path"].split("/")[-1]
            img_path = get_celeba_img_dir() / image_filename
            if img_path.exists():
                real_image_paths.append(img_path)
        
        print(f"\nExtracting features from {len(real_image_paths)} real images...")
        real_features = extract_features_with_inception(
            real_image_paths, 
            device="cuda", 
            batch_size=32
        )
        
        print(f"Extracting features from {len(gen_images)} generated images...")
        fake_features = extract_features_with_inception(
            gen_images, 
            device="cuda", 
            batch_size=32
        )
        
        if real_features is not None and fake_features is not None:
            fid_score = compute_fid_from_features(real_features, fake_features)
            
            if not np.isnan(fid_score):
                print(f"\n✅ FID Score: {fid_score:.2f}")
                
                # Save report
                results_root = get_results_root()
                reports_dir = results_root / "reports"
                reports_dir.mkdir(parents=True, exist_ok=True)
                
                fid_report = {
                    "fid_score": float(fid_score),
                    "real_samples": len(real_image_paths),
                    "generated_samples": len(gen_images),
                    "inference_steps": 50,
                    "guidance_scale": 7.5,
                }
                
                with open(reports_dir / "fid_report.json", 'w') as f:
                    json.dump(fid_report, f, indent=2)
                
                print(f"✓ Report saved")
                
                if fid_score < 50:
                    print("✅ GOOD: FID < 50")
                elif fid_score < 100:
                    print("⚠️  ACCEPTABLE: FID < 100")
                else:
                    print("❌ NEEDS IMPROVEMENT: FID > 100")
    
    except Exception as e:
        print(f"⚠️  FID computation failed: {e}")
        import traceback
        traceback.print_exc()
else:
    print(f"⚠️  Not enough images")

print("="*70 + "\n")


FID EVALUATION - CelebA Single Domain

Loading validation dataset for FID reference...
Validation images available: 20261
Generated images: 0
⚠️  Not enough images

